[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/diode.ipynb)

# A p–n Junction

This example computes a p–n junction by drift and diffusion: a bar of silicon doped on one side with acceptors and on the other with donors, its junction forming as the two sides touch, its current following a bias as Shockley's diode law has it, and under light a solar cell. Acceptors take electrons from the crystal and leave mobile holes behind; donors give it electrons. Where the two sides meet, electrons spill from the n side into the p side and holes the other way, and recombine there, each pair handing its energy, the band gap, to the lattice as heat. The dopants they leave behind are charged, and their field across the depletion region, emptied of carriers, holds the rest back, until each carrier's electrochemical potential, its Fermi level, is level across the bar and the bands bend by the built-in voltage. A forward bias lowers that barrier and the current grows exponentially with it; a reverse bias raises it, and the current saturates at the little that thermal generation supplies. Light makes electrons and holes in pairs, which the junction's field sweeps apart as a photocurrent against the diode's own: the solar cell.

The algebra is the bar's line in spacetime joined with four directions that have no metric, for electrons, holes, heat and light, `Algebra("x+t-") * Algebra("n0p0q0l0")`. Charge rides time: an electron is modelled as a unit along `n` with a negative charge, `mv.n.dual() - time`, a hole as a unit along `p` with a positive one, and the dopants' fixed charge lies along time alone. The potentials at each vertex are one plane, the electric potential along `mv.t` and each carrier's chemical potential along its own direction, and the plane reads a carrier's electrochemical potential, its quasi-Fermi level, by the regressive product, `potentials & electron`; the band diagram is that plane and the energy plane read on an electron and on a hole. Their drive, `-(gradients ^ potentials)`, pushes each carrier by what it holds, so that drift and diffusion are one drive; a carrier drifts along the force the drive puts on it, `drive.commutator(electron)`, and the material map that makes the drive the carriers' flux sweeps each along its drift, the same in every direction. The drive read on the flux it drives is the heat that transport makes, never negative. Recombination is a direction among the amounts, `gap * heat - (electron + hole)`: the energy plane pairs with it to nothing, and the potentials to minus the quasi-Fermi levels' splitting, whose sign the rate of mass action shares, so that recombination too produces entropy and never destroys it. The solar cell runs the same direction backward, its energy drawn from the light.

In the notation of semiconductor device physics, with $\psi$ the electric potential, $n$ and $p$ the densities of electrons and holes, $E_{Fn}$ and $E_{Fp}$ their quasi-Fermi levels, $N_D^+ - N_A^-$ the dopants' charge and $G$ the light's generation, the drift–diffusion model is Poisson's equation $-\nabla\cdot(\epsilon\nabla\psi) = q(p - n + N_D^+ - N_A^-)$ with the currents $\mathbf J_n = \mu_n n\nabla E_{Fn}$ and $\mathbf J_p = \mu_p p\nabla E_{Fp}$ and their continuity, $q\,\partial_t n = \nabla\cdot\mathbf J_n - q(R - G)$ and $q\,\partial_t p = -\nabla\cdot\mathbf J_p - q(R - G)$, here with recombination by mass action, $R = k(np - n_i^2)$. The ideal diode's current is Shockley's $J = J_0\,(e^{qV/kT} - 1)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import math
from collections.abc import Iterator
from functools import reduce
from operator import and_

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_states
from examples.thermodynamics.semiconductor_devices import render, scenarios

# The bar's line in spacetime, and the electrons, the holes, the heat and the light a cell holds or trades: each a
# direction of its own, degenerate, since nothing measures an amount against a length or a time.
ga = Algebra("x+t-") * Algebra("n0p0q0l0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The potentials, the electric potential along t and the carriers' chemical potentials along n and p; and their
# drive, a slope along the bar wedged with what each acts on.
Potentials = ga.gatype.from_blades("t n p")                                   # Vector+
Drive = ga.gatype.from_blades("xt xn xp")                                     # Bivector+
# A unit of charge, along time; an electron, a unit of the electrons' count with a negative charge, and a hole, a unit
# of the holes' count with a positive one; a unit of heat, and of the light's energy.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
heat = mv.q.dual()                                                            # [] Point-
light = mv.l.dual()                                                           # [] Point-
# An electron and a hole, what recombination takes and absorption makes.
pairs = electron + hole                                                       # [] Point-
# The carriers; their counts, which their chemical potentials read, the planes that read each count off an amount,
# and the carriers' charges, along time.
carriers = stack((electron, hole))                                            # [carriers] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [carriers] Point-
counting = stack((mv.n, mv.p))                                                # [carriers] Plane+
charges = carriers - counts                                                   # [carriers] Point-

## The bar of silicon

The bar is silicon at room temperature, a chain of short edges along it, doped as heavily with acceptors on its left half as with donors on its right. Lengths are in the doping's Debye length, the distance over which its carriers screen a charge, about 40 nm at 10¹⁶ dopants a cubic centimetre; energies and potentials are in the thermal energy, about 26 meV; densities are in the doping's; and times are in the time an electron takes to diffuse across a Debye length, also the time an excess of charge takes to relax away. In these units the intrinsic density, the electrons and holes that undoped silicon holds, is a millionth of the doping, and the band gap 43.6 thermal energies. A field varies along each edge by its ends' hats, and a hat's gradient is the facet facing its corner, here the other end, joined with time and every amount, which a field over space does not vary along, over the same join closed by the step to the corner. The drive across each edge is then one map of the potentials at its ends, `driving`, and read backwards across the regressive product, its adjugate gathers what flows along the edges into what flows out of each vertex's cell. The adjugate pairs the amount on the left of the potentials, where the potentials read amounts from the left, as `potentials & electron` does, and an amount read from the right is its grade involute read from the left, `amount & potentials == potentials & amount.involute()`: the outflow is `driving.adjugate().involute()`. In the bar's algebra, of an even count of directions, the involute turns an amount's sign.

In [ ]:
# Silicon at room temperature: its intrinsic density, a millionth of the doping; the band gap in thermal units, 1.12 eV;
# the electrons' and the holes' mobilities; and the permittivity, one in these units.
intrinsic, gap = 1e-6, 43.6
mobilities, permittivity = np.array([1.0, 1 / 3]), 1.0

In [ ]:
# The bar: its chain of edges, its vertices, and its doping, the acceptors' negative charge on the p side, left, and
# the donors' positive one on the n side, right, in units of the doping; the dopants' fixed charge lies along time.
topology, vertices, doping = scenarios.bar()
dopants = (time * doping).field()                                             # Point-[vertices]
corners = topology.incidence[-1, 0] * vertices                                # [corners] Point-[edges]
# Time and every amount, joined, which a field along the bar does not vary along.
unread = time & counts[0] & counts[1] & heat & light                          # [] Span-
# The corners of the facet facing each corner, here the other end, the first of them a step from the corner.
facing = corners[topology.complements[-1, 0]]                                 # [facet corners, corners] Point-[edges]
steps = corners - facing[0]                                                   # [corners] Point-[edges]
# The facet facing each corner, joined with time and every amount, a facet of one point with no edges of its own to
# join; over the same join closed by the step to the corner, it is the gradient that reads one from the facet to it.
facets = reduce(and_, facing[1:] - facing[0], unread)                         # [corners] Plane-[edges]
gradients = facets / (facets & steps)                                         # [corners] Plane+[edges]
# The integrals weigh by each edge's volume, its length, an ordinary number: the join of its edges from its first
# corner, with time and every amount, over the factorial of its dimension. A vertex's cell is a share of each edge
# beside it, one over the edge's count of corners.
joins = reduce(and_, corners[1:] - corners[0], unread)                        # Scalar-[edges]
volumes = joins.norm() / math.factorial(topology.n_dim)                       # Scalar+[edges]
corner_mean = topology.incidence[-1, 0].mean(axis=0)                          # Scalar+[edges <- vertices]
cells = ~corner_mean * volumes                                                # Scalar+[vertices]
# The drive across each edge of the potentials on its ends: minus their slope, their corners' values on their hats'
# gradients, wedged with what each acts on; and what flows out of each vertex's cell of what flows along the edges
# beside it, the drive's map read backwards, its amounts read from the left.
slopes = (spdiag(gradients) * topology.incidence[-1, 0]).sum(axis=0)          # Plane+[edges <- vertices]
driving = -(slopes ^ Potentials)                                              # (Drive+ <- Potentials+)[edges <- vertices]
outflow = driving.adjugate().involute()                                       # (Point- <- Flux-)[vertices <- edges]

In [ ]:
def carrier_densities(potentials: Potentials) -> Scalar:
    """The electrons and the holes per length at these potentials `[carriers] Scalar+[vertices]`: the intrinsic
    density times the exponential of each carrier's chemical potential, Boltzmann's distribution."""
    return intrinsic * (potentials & counts).exp()


def energy(potentials: Potentials) -> Plane:
    """The plane reading a parcel's energy at these potentials `Plane+[vertices]`: its charge at the electric
    potential, each electron at the conduction band's edge and each hole at the valence band's, a gap apart with the
    middle of the gap at nothing, and heat and light at their own energy."""
    return (potentials & time) * mv.t + gap / 2 * (mv.n + mv.p) + mv.q + mv.l


def bands(potentials: Potentials) -> Scalar:
    """The conduction band's edge, the valence band's, and the electrons' and the holes' quasi-Fermi levels at these
    potentials `[4] Scalar+[vertices]`, the holes' read as the electrons missing."""
    energies = energy(potentials)                                             # Plane+[vertices]
    return stack((energies & electron, -(energies & hole), potentials & electron, -(potentials & hole)))


# The slabs before they touch, at no electric potential: each carrier's chemical potential, the logarithm of its
# density over the intrinsic one, its majority carriers balancing its dopants and its minority carriers in equilibrium
# with them.
chemical = scenarios.slabs(doping)                                            # [carriers, vertices]
slab = (0 * mv.t + mv.n * chemical[0] + mv.p * chemical[1]).field()           # Potentials+[vertices]
# The band diagram's window of energy, in thermal units.
window = (-26.0, 60.0)
render.draw_bands(vertices, bands(slab), carrier_densities(slab), intrinsic, window);

The slabs before they touch, the p side on the left: the conduction band's edge above and the valence band's below, each band tinted by the carriers it holds, electrons blue and holes red, darker the more there are, on a logarithmic scale from the intrinsic density to the doping's; the electrons' quasi-Fermi level dashed blue and the holes' dashed red, their dashes alternating where the two are one. The bands are flat, and each slab's Fermi level sits near the band its majority carriers fill, the two a built-in voltage apart.

## The junction forming

The slabs touch, and a wire between their ends closes, bringing the p end to the n end's Fermi level, so that the ends become ohmic contacts held at their potentials: the p end's electric potential falls by the built-in voltage, the sum of the majority carriers' chemical potentials at the two ends. Each step then balances, in every cell, the electrons and holes it holds against what it held, what their fluxes bring and what recombines, all at the step's end: an implicit step, solved by Newton's method. Along time the cells' balances of the electrons and of the holes only repeat each other, and Gauss's law takes that place, the field's outflow from each cell against the charge it holds, so that each step also solves Poisson's equation. Newton's map is built of the same pieces as the balance: the carriers' capacitance, and the conductance and the permittivity pulled back through the drive's adjugate, a positive map on the potentials. The last of the cells below takes about five seconds, most of it drawing the animation.

In [ ]:
# The junction forming: the time between frames and the frames; the minority carriers' lifetime, one over the doping
# times the rate constant of recombination; and how far a Newton's step may change a chemical potential, in thermal
# units, and how little the last one changes them.
dt, n_frames, frame_duration_ms = 0.7, 100, 40
rate_constant = 1 / 2.0
limit, tolerance = 2.0, 1e-6

In [ ]:
# A carrier drifts along the force a drive puts on it, and its flux is the carrier swept along that drift: each
# carrier's conductance per unit of its density, the same in every direction.
conducting = mobilities * Drive.commutator(carriers).regressive(carriers)     # [carriers] Flux- <- Drive+
# The field's displacement under a drive: a unit of charge swept along the force the drive puts on it.
displacement = permittivity * Drive.commutator(time).regressive(time)         # [] Flux- <- Drive+


def run(potentials: Potentials, held: np.ndarray, contacts: Potentials, generations: np.ndarray,
        dt: float) -> Iterator[Potentials]:
    """The potentials `Potentials+[vertices]` from these and after each frame's step of `dt`: the vertices `held` at
    that frame's potentials of the contacts `[frames] Potentials+[held]`, the light making that frame's generation of
    pairs per length and time `[frames]`, each step solved by Newton's steps until one changes no chemical potential by
    more than the tolerance; at an infinite step, the steady state."""
    holding = topology.reading(held)                                          # Scalar+[held <- vertices]
    free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), held))   # Scalar+[free <- vertices]
    yield potentials
    for contact, generation in zip(contacts, generations):
        previous_parcels = (carrier_densities(potentials) * carriers).sum(axis=0)   # Point-[vertices]
        # The contacts take their potentials; the rest start from where they were.
        potentials = ~free * (free * potentials) + ~holding * contact         # Potentials+[vertices]
        while True:
            densities = carrier_densities(potentials)                         # [carriers] Scalar+[vertices]
            parcels = (densities * carriers).sum(axis=0)                      # Point-[vertices]
            drive = driving(potentials)                                       # Drive+[edges]
            conductance = ((corner_mean * densities) * conducting).sum(axis=0)   # (Flux- <- Drive+)[edges]
            # Recombination by mass action, less the light's generation, pairs per length and time.
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2) - generation   # Scalar+[vertices]
            # Gauss's law, along time: the field's outflow from each cell against the charge it holds.
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[vertices]
            gauss = outflow(volumes * displacement(drive)) - cells * charge  # Point-[vertices]
            # What each cell holds of the carriers, against what it held, what their fluxes take out and what
            # recombines; along time this only repeats the electrons' and the holes' balance, which Gauss's law
            # completes there.
            residual = (cells * (parcels - previous_parcels) / dt + outflow(volumes * conductance(drive))
                        + cells * rates * pairs + gauss)                      # Point-[vertices]
            # Newton's map: the carriers' capacitance over the step and their charge's in Gauss's law; the response
            # of the carriers' fluxes and of the field through the drive, and of the fluxes through the densities
            # on each edge, by each carrier's flux per unit of its density, gathered into each cell's outflow; and
            # recombination's, by the pairs' dyad.
            capacitance = spdiag(cells * (densities * (carriers / dt - charges) * (Potentials & counts)).sum(axis=0))
            through_drive = spdiag(volumes * (conductance + displacement))(driving)
            through_densities = (spdiag(volumes * conducting(drive)) * corner_mean
                                 * spdiag(densities * (Potentials & counts))).sum(axis=0)
            recombining = spdiag(cells * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs)))
            newton = capacitance + recombining + outflow(through_drive + through_densities)
            # Each carrier's balance taken per carrier, over its density, the charge's as it is: a carrier far scarcer
            # than the other, by many powers of e, still weighs alike in the solve.
            per_carrier = spdiag(time * (mv.t & Point) + (counts * (counting & Point) / densities).sum(axis=0))
            solving = free * per_carrier(newton) * ~free                      # (Point- <- Potentials+)[free <- free]
            change = -(~free * solving.solve(free * per_carrier(residual)))   # Potentials+[vertices]
            # Newton's step scaled down at each vertex where it would change the chemical potentials there by much
            # more than the limit: the densities follow them exponentially, and a full step from far off overshoots by
            # powers of e. The residual is linear in the electric potential, whose change needs no holding back.
            changes = (counts & change).abs().sum(axis=0)                     # Scalar+[vertices]
            potentials = potentials + change / (1 + changes / limit)
            if changes.max() < tolerance:
                break
        yield potentials

In [ ]:
# The wire brings the p end to the n end's Fermi level: its electric potential falls by the built-in voltage, the sum
# of the majority carriers' chemical potentials at the two ends.
ends = np.array([0, topology.n_vertices - 1])
built_in = chemical[1, 0] + chemical[0, -1]
wired = (mv.t * np.array([-built_in, 0.0]) + mv.n * chemical[0, ends]
         + mv.p * chemical[1, ends]).field()                                  # Potentials+[ends]
formed = show_states(run(slab, ends, np.ones(n_frames) * wired, np.zeros(n_frames), dt),
                     render.animate_forming, vertices, bands, carrier_densities, intrinsic, window,
                     name="diode_forming", duration_ms=frame_duration_ms)

The wire's voltage first spreads evenly along the bar and tilts the bands, as along any conductor; within a few relaxation times the majority carriers carry it to the junction, where the tilt gathers. Electrons spill into the p side and holes into the n side, their quasi-Fermi levels parting from the majority's; the depletion region empties, its tints fading, and the charge of the dopants it leaves bare bends the bands. The spilled carriers recombine, and the quasi-Fermi levels close, until the Fermi level is one level across the bar and the bands are bent by the built-in voltage.

## The current under a bias

A battery between the contacts raises the p end's electric potential by its bias, which sweeps from nothing forward, back through nothing into reverse, and back. Each frame is the steady state at its bias, a step of infinite length, at which each cell's carriers balance what flows and what recombines alone. The voltage across the bar is the rise of the electrons' quasi-Fermi level from contact to contact, and the current is the charge the carriers' fluxes bring into the n end's cell, against what flows out of it, `mv.t & outflow(volumes * flux)` there; in a steady state as much flows along every edge. This cell takes about five seconds, most of it drawing the animation.

In [ ]:
# The sweep: the bias at each frame, from nothing forward, back through nothing into reverse and back, in thermal
# units; the volt in thermal units, at 300 K; and the current's window, in units of the doping's charge times an
# electron's diffusion across a Debye length.
biases = scenarios.sweep()                                                    # [frames]
n_sweep_frames = len(biases)
volt, current_window = 1 / 0.02585, (-1.35e-4, 1.35e-4)
contacts = wired + (mv.t * np.stack([biases, 0 * biases], axis=-1)).field()   # [frames] Potentials+[ends]
# The rise of a field from the p end to the n end, and the n end's contact, which reads a field there.
rise = topology.reading(ends[1:]) - topology.reading(ends[:1])                # Scalar+[1 <- vertices]
n_contact = topology.reading(ends[1:])                                        # Scalar+[1 <- vertices]


def terminal(potentials: Potentials) -> tuple[Scalar, Scalar]:
    """The voltage across the bar at these potentials `[] Scalar+`, the rise of the electrons' quasi-Fermi level from
    contact to contact, and the current along it `[] Scalar+`, the charge the carriers' fluxes bring into the n end's
    cell."""
    conductance = ((corner_mean * carrier_densities(potentials)) * conducting).sum(axis=0)   # (Flux- <- Drive+)[edges]
    outflowing = outflow(volumes * conductance(driving(potentials)))          # Point-[vertices]
    return (rise * (potentials & electron)).sites.sum(), -(n_contact * (mv.t & outflowing)).sites.sum()


dark = show_states(run(formed[-1], ends, contacts, np.zeros(n_sweep_frames), np.inf),
                   render.animate_sweep, vertices, bands, carrier_densities, intrinsic, terminal, window,
                   (biases.min(), biases.max()), current_window, volt, render.dark_colour, [], name="diode_dark",
                   duration_ms=frame_duration_ms)

Beside the band diagram, the chart traces the current against the voltage, in volts, as the bias sweeps. Forward, the bands' bending falls by the bias and the quasi-Fermi levels part across the junction; the carriers each side injects into the other recombine within a diffusion length or two, their quasi-Fermi levels sloping back to the majority's, and the current grows by a factor e for each thermal volt, tenfold every 60 mV, shooting up near 0.4 V. In reverse the bending grows by the bias, the depletion region widens, and the current is the little that thermal generation within it supplies, far too little to show on this scale: the diode conducts one way.

## The solar cell

Light makes electrons and holes in pairs, at a rate its intensity sets. Absorption is recombination run backward, its energy drawn from the light rather than from the lattice's heat, `pairs - gap * light`, and the energy plane pairs with it to nothing as well. The light's energy arrives at the sun's temperature, and its free energy is that energy less the share the temperature takes along: absorption produces entropy as long as the quasi-Fermi levels part by less than the gap times the Carnot factor, `1 - 1 / sun`. The light comes on at no bias, the contacts shorted, and the bias then sweeps as before. This cell takes about five seconds, most of it drawing the animation.

In [ ]:
# The light: the pairs it makes per length and time, the frames over which it comes on at no bias, and the sun's
# temperature, in units of the lattice's.
generation_rate, n_lighting_frames = 1e-5, 15
sun = 5800 / 300
lighting = np.linspace(1 / n_lighting_frames, 1, n_lighting_frames)           # [lighting]
generations = generation_rate * np.concatenate([lighting, np.ones(n_sweep_frames)])   # [frames]
lit_biases = np.concatenate([0 * lighting, biases])                           # [frames]
lit_contacts = wired + (mv.t * np.stack([lit_biases, 0 * lit_biases], axis=-1)).field()   # [frames] Potentials+[ends]
lit = show_states(run(dark[-1], ends, lit_contacts, generations, np.inf), render.animate_sweep,
                  vertices, bands, carrier_densities, intrinsic, terminal, window, (biases.min(), biases.max()),
                  current_window, volt, render.lit_colour, [dark], name="diode_lit", duration_ms=frame_duration_ms)

The light's minority carriers lift the electrons' quasi-Fermi level on the p side and lower the holes' on the n side. At no bias the junction's field sweeps them across, a current against the diode's own, the photocurrent; the lit curve, orange, is the dark one, faint, shifted down by it, a little more in reverse, where the wider depletion region gathers more of the light's pairs. Between no bias and the open-circuit voltage, where the current falls to nothing, the current runs against the voltage and the bar delivers power: a solar cell. At open circuit the quasi-Fermi levels part across the whole bar by the open-circuit voltage, far below the Carnot bound, 41 thermal energies here.

In the notation of photovoltaics the lit curve is the dark one shifted by the photocurrent $J_L$, $J = J_0\,(e^{qV/kT} - 1) - J_L$, the open-circuit voltage is $V_{oc} = (kT/q)\ln(1 + J_L/J_0)$, and the splitting of the quasi-Fermi levels under light at the gap's energy is bounded, $E_{Fn} - E_{Fp} \le E_g\,(1 - T/T_\odot)$.

In [ ]:
# checks
recombination = gap * heat - pairs                                            # [] Point-
absorption = pairs - gap * light                                              # [] Point-
# Recombination and absorption keep the energy: the energy plane pairs with each to nothing.
for state in (formed[-1], lit[-1]):
    np.testing.assert_allclose((energy(state) & recombination).to_array(), 0.0, atol=1e-12)
    np.testing.assert_allclose((energy(state) & absorption).to_array(), 0.0, atol=1e-12)
# Gauss's law at every state: away from the contacts, the field's outflow from each cell is the charge it holds.
inner = topology.reading(topology.interior[0])
for state in formed + dark + lit:
    field_outflow = outflow(volumes * displacement(driving(state)))           # Point-[vertices]
    charge = cells * ((carrier_densities(state) * charges).sum(axis=0) + dopants)   # Point-[vertices]
    np.testing.assert_allclose((mv.t & (inner * (field_outflow - charge))).to_array(), 0.0, atol=1e-9)
# In equilibrium, at no bias in the dark, the Fermi level is one level across the bar, and the current is less than a
# thousandth of the current a thermal volt forward.
voltages, currents = (stack(readings).to_array() for readings in zip(*(terminal(state) for state in dark)))
fermi_levels = bands(dark[1])[2:].batch().to_array()                          # [2, vertices]
assert np.abs(fermi_levels - fermi_levels[0, -1]).max() < 1e-9
peak = voltages.argmax()
assert abs(currents[1]) < 1e-3 * np.interp(1.0, voltages[:peak], currents[:peak])
# Forward, the current grows by a factor e for each thermal volt; in reverse it saturates.
forward = (voltages > 4) & (voltages < 14)
growth = np.diff(np.log(currents[forward])) / np.diff(voltages[forward])
assert np.all((growth > 0.9) & (growth < 1.1))
reverse = -currents[voltages < -3]
assert reverse.max() < 1.15 * reverse.min()
# Transport, recombination and absorption never produce negative entropy, at any state of the three runs, down to the
# round-off of the largest production.
productions = []
lit_generations = np.concatenate([[0.0], generations])                         # [states]
for state, generation in [(state, 0.0) for state in formed + dark] + list(zip(lit, lit_generations)):
    densities = carrier_densities(state)                                      # [carriers] Scalar+[vertices]
    drive = driving(state)                                                    # Drive+[edges]
    conductance = ((corner_mean * densities) * conducting).sum(axis=0)        # (Flux- <- Drive+)[edges]
    # The entropy's slope over a parcel's amounts: the energy plane less the potentials, the light's free energy being
    # its energy less the share the sun's temperature takes along.
    entropy_slope = energy(state) - state - (1 - 1 / sun) * mv.l              # Plane+[vertices]
    rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2)    # Scalar+[vertices]
    productions.append([(volumes * (drive & conductance(drive))).batch().to_array(),
                        (cells * rates * (entropy_slope & recombination)).batch().to_array(),
                        (cells * generation * (entropy_slope & absorption)).batch().to_array()])
for process in zip(*productions):
    assert min(entropy.min() for entropy in process) > -1e-12 * max(entropy.max() for entropy in process)
# Under light the quasi-Fermi levels part by less than the Carnot bound.
splittings = [(bands(state)[2] - bands(state)[3]).batch().to_array().max() for state in lit]
assert max(splittings) < gap * (1 - 1 / sun)